# CE541E08 — Unit 4 · Day 33 — apply() and map()
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 33 of 45 |
| **Topics** | apply(axis=1) · map() · custom functions · SCS-CN via apply |
---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 33"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Applying Functions to DataFrames

`apply()` and `map()` are the two ways to apply a Python function to Pandas data:

| Method | Works on | What it does |
|---|---|---|
| `series.apply(func)` | One column (Series) | Applies func to each element |
| `df.apply(func, axis=1)` | Whole DataFrame | Applies func to each row (receives a Series) |
| `series.map(dict)` | One column | Replaces values using a dictionary |

These bridge the gap between Pandas and custom Python logic.

---
## Code Block 1 — apply(axis=1): Rational Method per Row

### What this code does

We define a function that computes peak discharge using the Rational Method, then use `apply(axis=1)` to run it for every row of the catchment DataFrame — adding two new columns Q_m3s and Q_Ls.

### Why each step is taken

**`def rational_Q(row)`:**
The function receives one row of the DataFrame as a Pandas Series. We access values by column name — `row['Area_ha']`, `row['C_runoff']`, etc. This is cleaner than writing the formula directly in the column assignment.

**`apply(rational_Q, axis=1)`:**
`axis=1` means "apply to each row". Pandas calls `rational_Q` once for every row, passing that row as a Series. The return value becomes the new column entry.

**Why `apply` instead of vectorised operation:**
For simple formulas, vectorised arithmetic is faster. But `apply` is preferred when the logic is complex (conditional, multi-step, or calls an external library) — such as the SCS-CN formula with the `if P > Ia` check.

### Algorithm

```
1. Define rational_Q(row):
   A = row['Area_ha'] * 10000       (ha to m²)
   i = row['Intensity_mmhr']/1000/3600  (mm/hr to m/s)
   C = row['C_runoff']
   return C * i * A

2. catchments.apply(rational_Q, axis=1)
   → calls rational_Q for each of the 5 rows
   → returns a Series of 5 Q values

3. Add as column: catchments['Q_m3s'] = ...
   catchments['Q_Ls'] = Q_m3s * 1000
```

### Expected output

```
  Catchment  Area_ha  C_runoff  Intensity_mmhr  Q_m3s    Q_Ls
0         A      125      0.65              52  0.1183  118.26
1         B       89      0.55              52  0.0712   71.20
2         C      234      0.70              52  0.2380  237.96
3         D      178      0.60              52  0.1554  155.43
4         E       95      0.50              52  0.0690   69.00
Total peak Q: 651.85 L/s
```

In [ ]:
import pandas as pd, numpy as np

catchments = pd.DataFrame({
    'Catchment'      : ['A','B','C','D','E'],
    'Area_ha'        : [125, 89, 234, 178, 95],
    'C_runoff'       : [0.65, 0.55, 0.70, 0.60, 0.50],
    'Intensity_mmhr' : [52, 52, 52, 52, 52],
})

def rational_Q(row):
    A = row['Area_ha'] * 10000           # ha → m²
    i = row['Intensity_mmhr'] / 1000 / 3600  # mm/hr → m/s
    C = row['C_runoff']
    return round(C * i * A, 4)

# axis=1: apply function to each ROW (row passed as a Series)
catchments['Q_m3s'] = catchments.apply(rational_Q, axis=1)
catchments['Q_Ls']  = catchments['Q_m3s'] * 1000

print(catchments)
print(f"Total peak Q: {catchments['Q_m3s'].sum()*1000:.2f} L/s")

### 🔁 Try this

Add a column `'Q_rank'` that ranks the catchments by Q_m3s from largest (1) to smallest.

Use `catchments['Q_m3s'].rank(ascending=False).astype(int)`

---
## Code Block 2 — map(): Replacing Codes with Descriptions

### What this code does

We use `.map()` with a dictionary to replace short codes (like 'A', 'B', 1, 2) with meaningful descriptions — soil type names and QA flag labels.

### Why each step is taken

**`df['Soil_code'].map(soil_map)`:**
`.map()` looks up each value in the dictionary and returns the corresponding value. If a key is not found, the result is `NaN`. This is cleaner than a series of `.replace()` calls or a loop.

**Two dictionaries — one for soil, one for QA:**
Each column gets its own mapping dictionary. The two `map()` calls create two new columns independently — they do not interfere with each other.

### Algorithm

```
1. Define soil_map: {'A':'Sandy loam', 'B':'Clay loam', 'C':'Clay'}
   Define qa_map: {1:'GOOD', 2:'SUSPECT', 3:'HIGH'}

2. df['Soil_type'] = df['Soil_code'].map(soil_map)
   → replaces A,B,C with descriptive names

3. df['QA_Flag'] = df['QA_code'].map(qa_map)
   → replaces 1,2,3 with text flags

4. Print selected columns
```

### Expected output

```
         Date  Flow_m3s              Soil_type  QA_Flag
0  2024-07-01     234.5  Sandy loam (low CN)     GOOD
1  2024-07-02     678.9  Clay loam (medium CN)   GOOD
2  2024-07-03    1234.5  Clay (high CN)          SUSPECT
3  2024-07-04     456.7  Clay loam (medium CN)   GOOD
4  2024-07-05     890.2  Sandy loam (low CN)     HIGH
```

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Date'     : ['2024-07-01','2024-07-02','2024-07-03','2024-07-04','2024-07-05'],
    'Flow_m3s' : [234.5, 678.9, 1234.5, 456.7, 890.2],
    'Soil_code': ['A','B','C','B','A'],
    'QA_code'  : [1, 1, 2, 1, 3],
})

# Dictionary mapping: code → description
soil_map = {'A':'Sandy loam (low CN)', 'B':'Clay loam (medium CN)', 'C':'Clay (high CN)'}
qa_map   = {1:'GOOD', 2:'SUSPECT', 3:'HIGH'}

# .map() replaces each value using the dictionary; unrecognised keys → NaN
df['Soil_type'] = df['Soil_code'].map(soil_map)
df['QA_Flag']   = df['QA_code'].map(qa_map)

print(df[['Date','Flow_m3s','Soil_type','QA_Flag']])

### 🔁 Try this

Add a new code `'D'` (Gravelly loam) to the soil_map dictionary.

Then change one row's Soil_code to 'D' and verify the map works.

What happens if you use a code that is NOT in the dictionary?

---
## Code Block 3 — apply with IMD Classification

### What this code does

We define a multi-condition IMD classification function and use `apply` to classify 30 days of rainfall into named categories.

### Why each step is taken

**Multi-condition function vs pd.cut:**
`pd.cut` (covered in Day 34) is simpler for fixed numeric bins. `apply` with a function is preferable when the logic is more complex — for example when the category depends on multiple columns, or when you need to return a message rather than just a label.

**`df['IMD_Cat'].value_counts()`:**
Counts how many days fall in each category. Because the categories are strings, `.value_counts()` sorts by frequency (most common first) by default.

### Algorithm

```
1. Define imd_category(rain):
   if rain == 0: return 'No rain'
   elif rain <= 15.5: return 'Light'
   elif rain <= 64.4: return 'Moderate'
   elif rain <= 115.5: return 'Heavy'
   elif rain <= 204.4: return 'Very Heavy'
   else: return 'Extremely Heavy'

2. df['IMD_Cat'] = df['Rainfall_mm'].apply(imd_category)
   → applies function to each value in column

3. df['IMD_Cat'].value_counts() → category counts
```

### Expected output

```
     Date  Rainfall_mm     IMD_Cat
0 2024-07-01         ...       ...
...

Category counts:
IMD_Cat
Light      ...
Moderate   ...
No rain    ...
Heavy      ...
```

In [ ]:
import pandas as pd, numpy as np

np.random.seed(1)
df = pd.DataFrame({
    'Date'       : pd.date_range('2024-07-01', periods=30, freq='D'),
    'Rainfall_mm': np.round(np.random.exponential(30, 30), 1),
})

def imd_category(rain):
    if rain == 0:          return 'No rain'
    elif rain <= 15.5:     return 'Light'
    elif rain <= 64.4:     return 'Moderate'
    elif rain <= 115.5:    return 'Heavy'
    elif rain <= 204.4:    return 'Very Heavy'
    else:                  return 'Extremely Heavy'

# .apply() on a Series: calls imd_category for each rainfall value
df['IMD_Cat'] = df['Rainfall_mm'].apply(imd_category)

print(df.head(10))
print()
print("Category counts:")
print(df['IMD_Cat'].value_counts())

### 🔁 Try this

Count the **total rainfall in each IMD category**:

`df.groupby('IMD_Cat')['Rainfall_mm'].sum().round(1)`

Which category contributed the most to the 30-day total?

---
## Code Block 4 — apply returning multiple columns

### What this code does

We define a pipe hydraulics function that returns two values (V and h_f) and use `apply` to compute both simultaneously for every pipe section — adding both as new columns in one call.

### Why each step is taken

**Returning `pd.Series({'V_ms':..., 'hf_m':...})`:**
When an apply function returns a Series with named keys, Pandas automatically creates multiple new columns from the result. This avoids running `apply` twice (once for V, once for h_f).

**`pd.concat([df, results], axis=1)`:**
`axis=1` concatenates columns side-by-side. `results` is a `(8,2)` DataFrame returned by the multi-column apply, and `df` is the original `(8,5)` DataFrame. The result is `(8,7)`.

### Algorithm

```
1. Define pipe_calc(row):
   Compute V using Manning's formula
   Compute hf using Darcy-Weisbach
   return pd.Series({'V_ms':..., 'hf_m':...})

2. results = df.apply(pipe_calc, axis=1)
   → (8,2) DataFrame: V and hf for each pipe

3. df = pd.concat([df, results], axis=1)
   → add both columns to the original DataFrame

4. df['OK'] = df['V_ms'].between(0.6, 3.0)
   → True if velocity in acceptable range
```

### Expected output

```
  Section  Diameter_mm  Length_m   Slope   V_ms   hf_m    OK
0       A          200       100   0.002  2.162  8.150  True
...
```

In [ ]:
import pandas as pd, numpy as np, math

df = pd.DataFrame({
    'Section'    : list('ABCDEFGH'),
    'Diameter_mm': [200,250,300,200,250,300,350,400],
    'Length_m'   : [100,150,200, 80,120,180,250,300],
    'Slope'      : [0.002,0.003,0.001,0.004,0.002,0.0015,0.001,0.001],
})

def pipe_calc(row):
    D = row['Diameter_mm']/1000
    n = 0.013; f = 0.018; g = 9.81
    R = D/4
    V  = (1/n) * R**(2/3) * row['Slope']**0.5
    hf = f * (row['Length_m']/D) * (V**2/(2*g))
    # Return a Series — Pandas creates one column per key
    return pd.Series({'V_ms': round(V,3), 'hf_m': round(hf,3)})

# apply returns a (8,2) DataFrame; concat adds it as columns
results = df.apply(pipe_calc, axis=1)
df = pd.concat([df, results], axis=1)
df['OK'] = df['V_ms'].between(0.6, 3.0)
print(df)

### 🔁 Try this

Find and print all pipe sections where `OK == False`.

Which sections need to be redesigned? What changes to diameter or slope would bring them into the acceptable range?

---
## Session Summary — apply() and map()

| Method | Syntax | What it does |
|---|---|---|
| Apply to column | `series.apply(func)` | func called once per element |
| Apply to rows | `df.apply(func, axis=1)` | func called once per row (row as Series) |
| Map codes | `series.map(dict)` | Replace values using dictionary |
| Multi-column return | Return `pd.Series({...})` in apply | Creates multiple new columns |
| Combine results | `pd.concat([df, results], axis=1)` | Add new columns to DataFrame |
| Between check | `series.between(lo, hi)` | True where lo <= value <= hi |
| Count categories | `series.value_counts()` | Frequency per unique value |

---
## Day 33 Assignment

Pipe network with 8 sections (from Code Block 4). Use `apply` to compute Manning's velocity and add a column `Status` = 'OK' if 0.6 ≤ V ≤ 3.0 m/s, else 'REVIEW'. Print a summary showing how many sections pass and fail.

### ▶ Assignment cell

In [ ]:
import pandas as pd, numpy as np, math

df = pd.DataFrame({
    'Section'    : list('ABCDEFGH'),
    'Diameter_mm': [200,250,300,200,250,300,350,400],
    'Length_m'   : [100,150,200, 80,120,180,250,300],
    'Slope'      : [0.002,0.003,0.001,0.004,0.002,0.0015,0.001,0.001],
})

def pipe_calc(row):
    D = row['Diameter_mm']/1000; n=0.013; f=0.018; g=9.81
    R = D/4
    V  = (1/n)*R**(2/3)*row['Slope']**0.5
    hf = f*(row['Length_m']/D)*(V**2/(2*g))
    return pd.Series({'V_ms':round(V,3),'hf_m':round(hf,3)})

results = df.apply(pipe_calc, axis=1)
df = pd.concat([df, results], axis=1)
df['OK'] = df['V_ms'].between(0.6, 3.0)
print(df)

---
- [ ] Run all cells — verify outputs
- [ ] Complete the assignment cell
- [ ] Upload: `Unit4_Pandas/CE541E08_U4_Day33.ipynb`
- [ ] Commit: `Day 33 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*